# 01. Preprocessing: BAM to PAT / Beta

Converts raw long-read BAM files to the methylation-call formats used throughout this project:
- `.pat.gz` — per-read CpG methylation patterns (wgbstools native format)
- `.beta`   — genome-wide uint8 (methylated_count, total_count) pairs at every CpG site

**Targets:** all 9 sequencing batches across ONT, PacBio Revio, and PacBio Sequel platforms.

**Mode:** `MODE = "TEST"` processes 3 samples for validation;
`MODE = "PRODUCTION"` runs the full batch sequentially.
For large-scale parallel preprocessing use **01b_dsub_preprocess.ipynb**.

**Requirements:** wgbstools binary + hg38 reference must be pre-installed (see `00_setup_environment.ipynb`).

## Cell 1 — Imports

In [ ]:
import os
import re
import time
import subprocess
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python environment ready.")

## Cell 2 — Configuration

Set `MODE = "TEST"` for a quick 3-sample end-to-end validation, or `"PRODUCTION"` to process all samples.
Set `ACTIVE_BATCH` to run a single batch, or `None` for all batches in sequence.

In [ ]:
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
PROJECT_ID       = os.environ["GOOGLE_PROJECT"]
HOME             = Path(os.getcwd())

# ── Mode ─────────────────────────────────────────────────────────
MODE = "TEST"   # "TEST" (3 samples) | "PRODUCTION" (all samples, skip_if_exists=True)
TEST_SAMPLE_LIMIT = 3

# ── wgbstools ────────────────────────────────────────────────────
TOOL_BIN        = HOME / "wgbs_tools" / "wgbstools"
CPG_CHROME_SIZE = HOME / "wgbs_tools" / "references" / "hg38" / "CpG.chrome.size"

# ── Local scratch ─────────────────────────────────────────────────
TMP = HOME / "tmp_preprocess"
TMP.mkdir(parents=True, exist_ok=True)

# ── Batch configuration ───────────────────────────────────────────
# BAM source paths and output GCS prefixes for all 9 sequencing batches
BATCH_CONFIG = {
    "bi_revio": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/Broad/revio/bam",
        "platform":   "PacBio_Revio",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/bi_revio",
    },
    "bi_sequel": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/Broad/sequel2e/bam",
        "platform":   "PacBio_Sequel",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/bi_sequel",
    },
    "bcm_revio": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/BCM/revio/bam",
        "platform":   "PacBio_Revio",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/bcm_revio",
    },
    "bcm_sequel": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/BCM/sequel2e/bam",
        "platform":   "PacBio_Sequel",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/bcm_sequel",
    },
    "bcm_ont": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/BCM/ont/bam",
        "platform":   "ONT",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/bcm_ont",
    },
    "jhu_ont": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/JHU/ont/bam",
        "platform":   "ONT",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/jhu_ont",
    },
    "uw_revio": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/UW/revio/bam",
        "platform":   "PacBio_Revio",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/uw_revio",
    },
    "uw_sequel": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/UW/sequel2e/bam",
        "platform":   "PacBio_Sequel",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/uw_sequel",
    },
    "ha_revio": {
        "bam_base":   "gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/HA/revio/bam",
        "platform":   "PacBio_Revio",
        "out_prefix": f"{WORKSPACE_BUCKET}/results/ha_revio",
    },
}

# ── Active batch selector ─────────────────────────────────────────
# Set to a key from BATCH_CONFIG to process one batch, or None to run all
ACTIVE_BATCH = None   # e.g. "bcm_ont"

if ACTIVE_BATCH is not None:
    assert ACTIVE_BATCH in BATCH_CONFIG, f"Unknown batch: {ACTIVE_BATCH}"
    ACTIVE_BATCHES = {ACTIVE_BATCH: BATCH_CONFIG[ACTIVE_BATCH]}
else:
    ACTIVE_BATCHES = BATCH_CONFIG

print(f"MODE             : {MODE}")
print(f"ACTIVE_BATCHES   : {list(ACTIVE_BATCHES.keys())}")
print(f"OUT root (sample): {WORKSPACE_BUCKET}/results/<batch_token>/")

## Cell 3 — BigQuery Cohort Query

Queries the All of Us Research Workbench database to retrieve all participants
with long-read whole-genome variant data (`has_lr_whole_genome_variant = 1`),
along with their date of birth (used to compute age at blood draw).

In [ ]:
cohort_sql = """
    SELECT
        person.person_id,
        person.birth_datetime         AS date_of_birth,
        person.sex_at_birth_concept_id,
        p_sex.concept_name            AS sex_at_birth,
        person.race_concept_id,
        p_race.concept_name           AS race,
        person.ethnicity_concept_id,
        p_ethnicity.concept_name      AS ethnicity
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.person` person
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex
        ON person.sex_at_birth_concept_id = p_sex.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race
        ON person.race_concept_id = p_race.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity
        ON person.ethnicity_concept_id = p_ethnicity.concept_id
    WHERE person.PERSON_ID IN (
        SELECT DISTINCT person_id
        FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person`
        WHERE has_lr_whole_genome_variant = 1
    )
"""

print("Querying cohort from BigQuery...")
cohort_df = pd.read_gbq(cohort_sql, project_id=PROJECT_ID, dialect="standard")
cohort_df["person_id"] = cohort_df["person_id"].astype(str)

print(f"Cohort size: {len(cohort_df):,} participants with long-read WGS")
display(cohort_df.head(3))

## Cell 4 — Validate wgbstools and Discover BAM Files

Checks that wgbstools and hg38 reference files are available, then scans each
configured batch folder on GCS to build a per-batch sample manifest.

In [ ]:
assert TOOL_BIN.exists(),        f"wgbstools not found: {TOOL_BIN}"
assert CPG_CHROME_SIZE.exists(), f"CpG.chrome.size not found: {CPG_CHROME_SIZE}"
assert shutil.which("bgzip"),    "bgzip not found — install: sudo apt-get install tabix"
print("wgbstools OK:", TOOL_BIN)
print("bgzip      OK\n")


def discover_bam_files(bam_base: str, project_id: str) -> list:
    "List all GRCh38-aligned BAM files under a batch BAM base directory."
    r = subprocess.run(
        ["gsutil", "-u", project_id, "ls", f"{bam_base}/"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  WARNING: cannot list {bam_base}: {r.stderr.strip()[:200]}")
        return []
    sample_dirs = [l.strip() for l in r.stdout.strip().split("\n") if l.strip()]
    bam_paths = []
    for d in sample_dirs:
        sid = d.rstrip("/").split("/")[-1]
        bam = f"{bam_base}/{sid}/GRCh38/{sid}.bam"
        bam_paths.append({"person_id": sid, "bam_path": bam})
    return bam_paths


# Build manifests for all active batches
batch_manifests = {}
for batch_name, cfg in ACTIVE_BATCHES.items():
    print(f"Scanning {batch_name} ...")
    records = discover_bam_files(cfg["bam_base"], PROJECT_ID)
    df = pd.DataFrame(records)
    # Merge with cohort to get date_of_birth
    df = df.merge(cohort_df[["person_id", "date_of_birth"]], on="person_id", how="left")
    batch_manifests[batch_name] = df
    print(f"  {len(df):4d} samples  platform={cfg['platform']}")

total = sum(len(v) for v in batch_manifests.values())
print(f"\nTotal samples across active batches: {total:,}")

## Cell 5 — Core Processing Function

`process_one_sample` runs the full per-sample pipeline:
1. Downloads the BAM from GCS (requester-pays)
2. Runs `wgbstools bam2pat` to produce a per-read CpG pattern file
3. Runs `wgbstools pat2beta` to produce genome-wide coverage/methylation arrays
4. Uploads `.pat.gz`, `.pat.gz.csi`, and `.beta` to the batch output prefix on GCS
5. Cleans up local files

In [ ]:
def gcs_exists(path: str) -> bool:
    "Return True if a GCS object exists."
    r = subprocess.run(["gsutil", "-q", "stat", path], capture_output=True)
    return r.returncode == 0


def process_one_sample(
    person_id:      str,
    bam_gcs_path:   str,
    out_prefix_gcs: str,
    threads:        int  = 8,
    skip_if_exists: bool = True,
) -> dict:
    "Run bam2pat + pat2beta for one sample and upload results to GCS."
    pat_gcs  = f"{out_prefix_gcs}/{person_id}.pat.gz"
    beta_gcs = f"{out_prefix_gcs}/{person_id}.beta"

    print(f"\n=== {person_id} ===")

    if skip_if_exists and gcs_exists(pat_gcs):
        print(f"  Skip (exists): {pat_gcs}")
        return {"person_id": person_id, "status": "skipped_exists"}

    local_bam  = TMP / f"{person_id}.bam"
    local_pat  = TMP / f"{person_id}.pat.gz"
    local_csi  = TMP / f"{person_id}.pat.gz.csi"
    local_beta = TMP / f"{person_id}.beta"

    t0 = time.time()

    try:
        # ── 1. Download BAM ───────────────────────────────────────
        print(f"  Downloading BAM: {bam_gcs_path}")
        ret = subprocess.run(
            ["gcloud", "storage", "cp", bam_gcs_path, str(local_bam),
             f"--billing-project={PROJECT_ID}"],
            check=True
        )

        # ── 2. bam2pat ────────────────────────────────────────────
        print(f"  Running bam2pat ...")
        subprocess.run(
            [str(TOOL_BIN), "bam2pat", str(local_bam),
             "--genome", "hg38", "--threads", str(threads), "-f"],
            check=True, cwd=str(TMP)
        )
        assert local_pat.exists(), f"bam2pat did not produce {local_pat}"

        # ── 3. Index PAT ──────────────────────────────────────────
        subprocess.run([str(TOOL_BIN), "index", str(local_pat)], check=True)

        # ── 4. pat2beta ───────────────────────────────────────────
        print(f"  Running pat2beta ...")
        subprocess.run(
            [str(TOOL_BIN), "pat2beta", "-f",
             "--genome", "hg38", "--threads", str(threads),
             "--out_dir", str(TMP), str(local_pat)],
            check=True
        )
        assert local_beta.exists(), f"pat2beta did not produce {local_beta}"

        # ── 5. Upload ─────────────────────────────────────────────
        print(f"  Uploading to {out_prefix_gcs}/")
        upload_files = [local_pat, local_csi, local_beta]
        subprocess.run(
            ["gsutil", "-m", "-q", "cp",
             *[str(p) for p in upload_files if p.exists()],
             f"{out_prefix_gcs}/"],
            check=True
        )

        elapsed = time.time() - t0
        print(f"  Done in {elapsed/60:.1f} min")
        return {"person_id": person_id, "status": "ok", "elapsed_min": round(elapsed/60, 2)}

    except Exception as e:
        print(f"  ERROR: {e}")
        return {"person_id": person_id, "status": "error", "error": str(e)}

    finally:
        for p in [local_bam, local_pat, local_csi, local_beta]:
            if p.exists():
                try: p.unlink()
                except: pass


print("process_one_sample defined.")

## Cell 6 — Batch Runner

In [ ]:
def run_batch(batch_name: str, max_samples: int = None, skip_if_exists: bool = True) -> pd.DataFrame:
    "Process all samples in a named batch."
    cfg = BATCH_CONFIG[batch_name]
    manifest = batch_manifests[batch_name].copy()

    if max_samples is not None:
        manifest = manifest.head(max_samples)

    print(f"\n{'='*55}")
    print(f"  BATCH: {batch_name}  ({len(manifest)} samples)")
    print(f"  OUT  : {cfg['out_prefix']}/")
    print(f"{'='*55}")

    rows = []
    for i, row in manifest.iterrows():
        print(f"\n[{i+1}/{len(manifest)}]")
        result = process_one_sample(
            person_id      = str(row["person_id"]),
            bam_gcs_path   = row["bam_path"],
            out_prefix_gcs = cfg["out_prefix"],
            skip_if_exists = skip_if_exists,
        )
        rows.append(result)

    stats = pd.DataFrame(rows)
    stats_path = TMP / f"{batch_name}_stats.csv"
    stats.to_csv(stats_path, index=False)
    subprocess.run(
        ["gsutil", "-m", "-q", "cp", str(stats_path),
         f"{cfg['out_prefix']}/{batch_name}_stats.csv"],
        check=True
    )
    print(f"\nStats saved to {cfg['out_prefix']}/{batch_name}_stats.csv")
    print(stats["status"].value_counts().to_string())
    return stats


print("run_batch defined.")

## Cell 7 — TEST Run (3 Samples from Active Batch)

Processes 3 samples end-to-end to validate the pipeline.
Set `MODE = "TEST"` in Cell 2.

In [ ]:
if MODE == "TEST":
    test_batch = list(ACTIVE_BATCHES.keys())[0]
    print(f"=== TEST RUN: {TEST_SAMPLE_LIMIT} samples from {test_batch} ===")
    stats_test = run_batch(test_batch, max_samples=TEST_SAMPLE_LIMIT, skip_if_exists=False)
    display(stats_test)
else:
    print("MODE='PRODUCTION' — skipping TEST cell.")

## Cell 8 — PRODUCTION Run (All Active Batches)

Processes all samples across all active batches sequentially.
Already-processed samples are skipped via the `skip_if_exists` check.

**Runtime:** ~90 min/sample for ONT (large BAMs); ~30 min/sample for PacBio.
For parallel processing, use **01b_dsub_preprocess.ipynb**.

Set `MODE = "PRODUCTION"` in Cell 2 and optionally set `ACTIVE_BATCH` to target one batch.

In [ ]:
if MODE == "PRODUCTION":
    all_stats = {}
    for batch_name in ACTIVE_BATCHES:
        print(f"\n{'#'*55}")
        print(f"# Starting batch: {batch_name}")
        print(f"{'#'*55}")
        stats = run_batch(batch_name, max_samples=None, skip_if_exists=True)
        all_stats[batch_name] = stats

    print("\n\n=== PRODUCTION SUMMARY ===")
    for bn, s in all_stats.items():
        ok  = (s["status"] == "ok").sum()
        ski = (s["status"] == "skipped_exists").sum()
        err = (s["status"] == "error").sum()
        print(f"  {bn:30s}: ok={ok:4d}  skipped={ski:4d}  error={err:4d}")
else:
    print("MODE='TEST' — skipping PRODUCTION cell.")

## Cell 9 — QC: Processing Time Histogram

Visualises per-sample processing time distribution across all completed samples.

In [ ]:
# Load all batch stats from GCS
all_rows = []
for batch_name, cfg in ACTIVE_BATCHES.items():
    stats_gcs = f"{cfg['out_prefix']}/{batch_name}_stats.csv"
    local_csv = TMP / f"{batch_name}_stats_qc.csv"
    try:
        subprocess.run(["gsutil", "-m", "-q", "cp", stats_gcs, str(local_csv)], check=True)
        df = pd.read_csv(local_csv)
        df["batch"] = batch_name
        df["platform"] = cfg["platform"]
        all_rows.append(df)
    except subprocess.CalledProcessError:
        print(f"  Stats not yet available for {batch_name}")

if all_rows:
    stats_all = pd.concat(all_rows, ignore_index=True)
    ok = stats_all[stats_all["status"] == "ok"]

    print(f"Total processed (ok): {len(ok)}")
    print(stats_all["status"].value_counts().to_string())

    if "elapsed_min" in ok.columns and len(ok) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        # Per-platform elapsed time
        for ax, group_col in zip(axes, ["platform", "batch"]):
            for grp, sub in ok.groupby(group_col):
                if "elapsed_min" in sub.columns:
                    ax.hist(sub["elapsed_min"].dropna(), bins=15, alpha=0.6, label=grp)
            ax.set_xlabel("Elapsed time (min)", fontsize=9)
            ax.set_ylabel("Samples", fontsize=9)
            ax.set_title(f"Processing time by {group_col}", fontsize=10)
            ax.legend(fontsize=7, frameon=False)

        plt.tight_layout()
        plt.show()

        print("\nMedian processing time by platform (min):")
        display(ok.groupby("platform")["elapsed_min"].median().round(1).reset_index())
else:
    print("No stats loaded. Run Cell 7 (TEST) or Cell 8 (PRODUCTION) first.")